# Knee v6: 5-fold retrain (set CONFIG in cell 0: "convnext" first, then "both")
tok  = `--head tokens` (per-label attention over every window).  both = tokens + canonical geometry / right-knee mirroring.
Same recipe as the baseline and v4mix 5-fold runs: B0, 160 px, 12 epochs, folds 0-2 on GPU 0 and 3-4 on GPU 1 (~3.7 h wall).
Outputs: `models_v6<CONFIG>/fold{i}_v6<CONFIG>.pt` (unique names, ready to sit next to the other 10 models), `oof_v6<CONFIG>.csv`.
Inputs: competition data, `knee-mri-pipeline` (knee_pipeline_v6), caching output, training output (oof_all.csv);
optional: the v4mix notebook output (for the 3-way OOF ensemble). GPU T4 x2, internet ON, Save & Run All.

In [1]:
import os, sys, json, subprocess, time, threading, shutil
from pathlib import Path
import numpy as np, pandas as pd

CONFIG = os.environ.get("KNEE_CONFIG", "convnext")          # "tok", "both" or "convnext"
assert CONFIG in ("tok", "both", "convnext")
TAG = f"v6{CONFIG}"
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA",   "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
BASE   = Path(os.environ.get("KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"))
V4MIX  = Path(os.environ.get("KNEE_V4MIX",  "/kaggle/input/notebooks/theokorir/knee-v4-mix-5fold/oof_v4mix.csv"))   # optional
for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    assert p.exists(), f"{name} not found: {p}"
for f in ("geom.py", "geom_table.py", "tests/smoke_v6.py"):
    assert (CODE / f).exists(), f"{f} missing: dataset does not hold knee_pipeline_v6"
print("CONFIG", CONFIG, "| v4mix OOF available:", V4MIX.exists())

CONFIG convnext | v4mix OOF available: True


In [2]:
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})\n{r.stdout[-800:]}")
    assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"
EXTRA = ["--head", "tokens"]
if CONFIG == "convnext":
    EXTRA += ["--backbone", "convnext_tiny.in12k_ft_in1k"]
if CONFIG == "both":
    t0 = time.time()
    r = subprocess.run([sys.executable, str(CODE / "geom_table.py"), "--split", "train", "--data_dir", str(DATA),
                        "--out", str(W / "geom.csv"), "--K", "1", "--workers", "16"], capture_output=True, text=True, cwd=str(CODE))
    print(r.stdout[-1800:], f"[{time.time()-t0:.0f}s]")
    assert r.returncode == 0 and (W / "geom.csv").exists(), r.stderr[-1500:]
    EXTRA += ["--geom_csv", str(W / "geom.csv")]
print("extra train args:", EXTRA)

== tests/test_geom.py (exit 0)
canonical orientation identical for 24 acquisition conventions
right/left mirror normalisation OK for 3 planes
laterality rules OK
ALL GEOM TESTS PASS

== tests/smoke_v6.py (exit 0)
head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
backbone convnext_tiny.in12k_ft_in1k: OK | vit_size None | params 28.9M
backbone efficientnet_b2: OK | vit_size None | params 8.9M
backbone vit_small_patch14_dinov2.lvd142m: OK | vit_size 168 | params 22.5M
backbone vit_small_patch16_224.dino: OK | vit_size 160 | params 22.6M
backbone deit_tiny_patch16_224.fb_in1k: OK | vit_size 160 | params 6.4M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS

extra train args: ['--head', 'tokens', '--backbone', 'convnext_tiny.in12k_ft_in1k']


In [3]:
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--epochs", "12", "--pretrained",
             "--workers", "2", "--train_slices", "16", "--bs", "4", "--accum", "2"]
JOBS = {0: (f"{TAG}_g0", ["0", "1", "2"]), 1: (f"{TAG}_g1", ["3", "4"])}
LOGS = {}
def run(gpu, name, folds):
    out = W / name
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); return
    cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + EXTRA + ["--only_folds"] + folds + ["--out", str(out)]
    LOGS[name] = W / f"{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    with open(LOGS[name], "w") as fh:
        rc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE)).wait()
    print(f"[{name}] exit code {rc}", flush=True)
threads = [threading.Thread(target=run, args=(g, n, f)) for g, (n, f) in JOBS.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(900)
    last = {}
    for n, f in LOGS.items():
        ls = [l for l in f.read_text().splitlines() if "ep " in l] if f.exists() else []
        last[n] = ls[-1] if ls else "(none yet)"
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished")

[v6convnext_g0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_tiny.in12k_ft_in1k --only_folds 0 1 2 --out /kaggle/working/v6convnext_g0
[v6convnext_g1] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_tiny.in12k_ft_in1k --only_folds 3 4 --out /kaggle/working/v6convnext_g1
[0.25 h] {
 "v6convnext_g0": "[f0] ep 1/12 loss 0.7984 | val macro-AUC 0.7002 | gold

In [4]:
# ---- collect: unique model names + merged OOF ----
M = W / f"models_{TAG}"; M.mkdir(exist_ok=True)
for name, folds in JOBS.values():
    for k in folds:
        src = W / name / f"fold{k}.pt"
        assert src.exists(), f"missing {src}: a job failed, do not use this output"
        shutil.copy(src, M / f"fold{k}_{TAG}.pt")
print(sorted(p.name for p in M.glob("*.pt")))
assert len(list(M.glob("*.pt"))) == 5
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
o0, o1 = pd.read_csv(W / JOBS[0][0] / "oof.csv"), pd.read_csv(W / JOBS[1][0] / "oof.csv")
oof = o0.copy()
for l in LAB:
    oof[f"p_{l}"] = o0[f"p_{l}"].fillna(o1[f"p_{l}"])
assert oof[[f"p_{l}" for l in LAB]].notna().all().all(), "some studies have no OOF prediction"
assert (o0.fold == o1.fold).all(), "fold assignment differs between the two jobs"
oof.to_csv(W / f"oof_{TAG}.csv", index=False)
print("saved", f"oof_{TAG}.csv", oof.shape)

['fold0_v6convnext.pt', 'fold1_v6convnext.pt', 'fold2_v6convnext.pt', 'fold3_v6convnext.pt', 'fold4_v6convnext.pt']
saved oof_v6convnext.csv (4407, 40)


In [5]:
# ---- pooled 5-fold analysis on the FIXED old labels (confident: y>=0.9 or <=0.1) ----
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
assert len(base) == len(oof) and np.allclose(base[[f"y_{l}" for l in LAB]].values, oof[[f"y_{l}" for l in LAB]].values, atol=1e-6), "fixed labels differ"
fold = oof.fold.values
def frank(df):                                    # rank within each fold (models are compared on rank, as at inference)
    out = {}
    for l in LAB:
        p = df[f"p_{l}"].values.astype(float); r = np.zeros(len(p))
        for k in np.unique(fold): m = fold == k; r[m] = rankdata(p[m]) / m.sum()
        out[l] = r
    return out
def auc_l(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len({*(y[m] >= 0.9)}) == 2 else np.nan
Y = {l: base[f"y_{l}"].values for l in LAB}
arms = {"baseline": base, TAG: oof}
if V4MIX.exists():
    v4 = pd.read_csv(V4MIX); assert len(v4) == len(base); arms["v4mix"] = v4
R = {n: frank(d) for n, d in arms.items()}
R["ens base+" + TAG] = {l: (R["baseline"][l] + R[TAG][l]) / 2 for l in LAB}
if "v4mix" in R:
    R["ens base+v4mix+" + TAG] = {l: (R["baseline"][l] + R["v4mix"][l] + R[TAG][l]) / 3 for l in LAB}
    R["ens base+v4mix"] = {l: (R["baseline"][l] + R["v4mix"][l]) / 2 for l in LAB}
T = pd.DataFrame({n: {l: auc_l(Y[l], r[l]) for l in LAB} for n, r in R.items()})
T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print("All 5 folds pooled with per-fold ranks:\n", T.round(3).to_string())
print("\nper-fold macro (raw predictions):")
for k in range(5):
    m = fold == k
    f = lambda df: np.nanmean([auc_l(Y[l][m], df.loc[m, f"p_{l}"].values) for l in LAB])
    print(f"  fold {k}: baseline {f(base):.4f} | {TAG} {f(oof):.4f} | delta {f(oof)-f(base):+.4f}")
g = oof.is_gold.values.astype(bool)
if g.sum() > 10:
    gm = lambda r: np.nanmean([roc_auc_score((Y[l][g] >= 0.5).astype(int), r[l][g]) for l in LAB if len({*(Y[l][g] >= 0.5)}) == 2])
    print(f"\ngold ({int(g.sum())} studies, CIs about +/-0.1, NOT decisive): " + " | ".join(f"{n} {gm(r):.3f}" for n, r in R.items()))
rng = np.random.default_rng(0); d = []
for _ in range(300):
    b = rng.integers(0, len(base), len(base))
    d.append(np.nanmean([auc_l(Y[l][b], R[TAG][l][b]) - auc_l(Y[l][b], R["baseline"][l][b]) for l in KEY4]))
print(f"\nKEY-4 {TAG} - baseline: {T.loc['KEY-4', TAG]-T.loc['KEY-4','baseline']:+.4f}  90% CI [{np.percentile(d,5):+.4f}, {np.percentile(d,95):+.4f}] (study resampling only; seed noise on one fold was ~0.005 KEY-4, ~0.01 for the v6 config)")

All 5 folds pooled with per-fold ranks:
                   baseline  v6convnext  v4mix  ens base+v6convnext  ens base+v4mix+v6convnext  ens base+v4mix
ACL                  0.781       0.853  0.804                0.840                      0.834           0.798
MCL                  0.725       0.797  0.716                0.774                      0.759           0.723
Medial Meniscus      0.770       0.881  0.772                0.848                      0.828           0.774
Lateral Meniscus     0.736       0.786  0.734                0.771                      0.762           0.738
Medial OA            0.826       0.860  0.826                0.852                      0.847           0.829
Lateral OA           0.747       0.777  0.740                0.772                      0.765           0.746
PF OA                0.717       0.727  0.717                0.732                      0.731           0.720
Effusion             0.822       0.840  0.808                0.838             